# Recipe variants and test-time augmentation on Kaggle: c-ResUNet-light

Ablations of the adopted training recipe, one change at a time (one run each, seed 42; checkpoint `best.pt` selected on the
validation images as for the adopted network), followed by test-time augmentation (TTA) of the adopted network.

| run | change from the adopted recipe | output |
|---|---|---|
| early stopping | stop after 15 epochs without improvement (`--patience 15`) | `runs/cresunet_light_early_stop/` |
| no gamma | brightness augmentation with gain only (`--no_gamma`) | `runs/cresunet_light_no_gamma/` |
| per-image normalization | statistics of each image instead of the training set (`--norm per_image`) | `runs/cresunet_light_per_image/` |
| wider network | 32 base channels instead of 16 (`--base_ch 32`) | `runs/cresunet_light_wide/` |
| 200 epochs | `--epochs 200 --patience 200` | `runs/cresunet_light_200ep/` |
| TTA | no training: mean probability over the 8 symmetries of the square, adopted network (`runs/cresunet_light/best.pt`), train / validation / test images | `runs/cresunet_light_tta/` |

After each training, predictions on the validation images with `best.pt` (`pred_val.npz`); the comparisons with the adopted
network are computed locally. The TTA test predictions are produced in the same run but evaluated only if TTA is adopted.
About 6 h on a T4. The code dataset must include `runs/cresunet_light/best.pt` (see `kaggle/README.md`).

Before running (the two input datasets are described in `kaggle/README.md`):
1. **Settings → Accelerator → GPU T4 x2**. Internet is not needed.
2. **Add Input**: the code dataset `cellseg-code` and the data dataset `fnc-green-data`.
3. Short check: `PROBE = True` (parameters cell) → **Run All**: the widest variant (checks the GPU memory) for 1 epoch on 8 training and 4 validation images, and TTA on the validation images only (~3 min).
4. Full run: `PROBE = False` → **Save Version → Save & Run All (Commit)**; the commit keeps running with the browser closed.
5. Download the output folders from the **Output** tab of the version into the repository's `runs/`.

In [ ]:
# 1) GPU available?
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Settings → Accelerator → 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Find code and data under /kaggle/input, copy the code to /kaggle/working (/kaggle/input is read-only)
import os
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def find_one(pattern):
    '''First path under INPUT matching the pattern (Kaggle can nest dataset folders in different ways).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} (were both datasets added with 'Add Input'?)"
    return hits[0]

CODE_SRC = find_one("models/c_res_unet_small_rf.py").parent.parent
IMAGES = find_one("green/trainval/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
TEST_IMAGES = find_one("green/test/images")
TEST_MASKS = find_one("cleaned_masks/test/Green/masks")
print("code:  ", CODE_SRC)
print("images:", IMAGES, "→", len(list(IMAGES.glob("*.png"))), "png (expected 212)")
print("masks: ", MASKS, "→", len(list(MASKS.glob("*.png"))), "png (expected 213: 128.png has no image)")

CODE = WORK / "code"
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)
os.chdir(CODE)
assert (CODE / "runs/cresunet_light/best.pt").exists(), "runs/cresunet_light/best.pt missing from the code dataset"
print("working dir:", os.getcwd())

## 3) Parameters

- `PROBE = True`: short check; `PROBE = False`: full run.
- `VARIANTS`: run name → options added to the adopted recipe (later options override earlier ones).

In [ ]:
PROBE = True

BASE = ["--small_rf", "--epochs", "100", "--patience", "100"]          # adopted recipe (c-ResUNet-light)
VARIANTS = {
    "cresunet_light_early_stop": ["--patience", "15"],
    "cresunet_light_no_gamma": ["--no_gamma"],
    "cresunet_light_per_image": ["--norm", "per_image"],
    "cresunet_light_wide": ["--base_ch", "32"],
    "cresunet_light_200ep": ["--epochs", "200", "--patience", "200"],
}
TTA_SPLITS = ["train", "val", "test"]
SPLIT_JSON = "splits/green_split.json"
LIMITS = []
if PROBE:
    VARIANTS = {"probe_cresunet_light_wide": ["--base_ch", "32"]}       # the widest network: checks the GPU memory
    TTA_SPLITS = ["val"]
    LIMITS = ["--epochs", "1", "--patience", "1", "--train_limit", "8", "--val_limit", "4"]
print(list(VARIANTS), "| TTA on", TTA_SPLITS)

## 4) Training and validation predictions, variant by variant

The loop stops at the first error.

In [ ]:
import subprocess
import sys
import time

def run(cmd):
    '''Runs a command, streams its output, stops on error.'''
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, f"failed (exit code {p.returncode})"

t_start = time.time()
for name, options in VARIANTS.items():
    t0 = time.time()
    out = WORK / "runs" / name
    run([sys.executable, "-m", "train.training", "--images_dir", IMAGES, "--masks_dir", MASKS, "--split_json", SPLIT_JSON,
         "--out_dir", out, "--num_workers", 4, *BASE, *options, *LIMITS])
    run([sys.executable, "-m", "evaluation.predict", "--ckpt", out / "best.pt", "--images_dir", IMAGES, "--masks_dir", MASKS,
         "--split_json", SPLIT_JSON, "--subset", "val", "--out", out / "pred_val.npz"])
    print(f"=== {name} done in {(time.time() - t0) / 60:.1f} min (total {(time.time() - t_start) / 60:.1f} min)", flush=True)

## 5) TTA of the adopted network (8 symmetries)

In [ ]:
tta_out = WORK / "runs" / ("probe_cresunet_light_tta" if PROBE else "cresunet_light_tta")
for split in TTA_SPLITS:
    images, masks = (TEST_IMAGES, TEST_MASKS) if split == "test" else (IMAGES, MASKS)
    run([sys.executable, "-m", "experiments.predict_tta", "--ckpt", "runs/cresunet_light/best.pt", "--images_dir", images,
         "--masks_dir", masks, "--split_json", SPLIT_JSON, "--subset", split, "--out", tta_out / f"pred_{split}.npz"])
print(f"total {(time.time() - t_start) / 60:.1f} min")

In [ ]:
# 6) Summary: one row per variant
import json
import numpy as np
import pandas as pd

val_names = json.load(open(SPLIT_JSON))["val"]
rows = []
for name in VARIANTS:
    out = WORK / "runs" / name
    log = pd.read_csv(out / "log.csv")
    cfg = json.load(open(out / "config.json"))
    pred = np.load(out / "pred_val.npz")
    assert cfg["small_rf"] and list(pred["names"]) == val_names
    rows.append({"run": name, "epochs": len(log), "best_epoch": int(log.epoch[log.val_dice_global.idxmax()]),
                 "best_dice": log.val_dice_global.max(), "base_ch": cfg["base_ch"], "norm": cfg["norm"],
                 "no_gamma": cfg["no_gamma"], "patience": cfg["patience"], "minutes": log.time_s.sum() / 60})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("TTA files:", sorted(p.name for p in tta_out.iterdir()))